In [1]:
// Añadimos Apache Spark Core y SQL para Scala 2.13
// El sufijo _2.13 es gestionado automáticamente por $ivy cuando usamos %%
import $ivy.`org.apache.spark::spark-core:4.1.1`
import $ivy.`org.apache.spark::spark-sql:4.1.1`

println("✅ Dependencias de Spark cargadas correctamente")

✅ Dependencias de Spark cargadas correctamente


import $ivy.$
import $ivy.$

In [2]:
import org.apache.log4j.{Level, Logger}

// Silenciamos los logs de Spark para que el output sea legible
Logger.getLogger("org").setLevel(Level.ERROR)
Logger.getLogger("akka").setLevel(Level.ERROR)

println("✅ Logs de Spark configurados (solo se mostrarán errores)")

✅ Logs de Spark configurados (solo se mostrarán errores)


import org.apache.log4j.{Level, Logger}

In [6]:
import org.apache.spark.sql.SparkSession

// Creamos la SparkSession en modo local
// "local[*]" significa: usa todos los núcleos de CPU disponibles
val spark = SparkSession.builder()
  .appName("IntroSpark")
  .master("local[*]")
  // ESTAS LÍNEAS SON LAS QUE FALTAN PARA JAVA 17:
  .config("spark.driver.extraJavaOptions", "--add-opens=java.base/java.nio=ALL-UNNAMED --add-opens=java.base/sun.nio.ch=ALL-UNNAMED")
  .config("spark.executor.extraJavaOptions", "--add-opens=java.base/java.nio=ALL-UNNAMED --add-opens=java.base/sun.nio.ch=ALL-UNNAMED")
  .config("spark.ui.showConsoleProgress", "false")
  .getOrCreate()

val sc = spark.sparkContext

println(s"✅ SparkSession creada correctamente")
println(s"   Versión de Spark: ${spark.version}")
println(s"   Nombre de la app: ${sc.appName}")
println(s"   Master:           ${sc.master}")
println(s"   Spark UI:         http://localhost:4040")

✅ SparkSession creada correctamente
   Versión de Spark: 4.1.1
   Nombre de la app: IntroSpark
   Master:           local[*]
   Spark UI:         http://localhost:4040


import org.apache.spark.sql.SparkSession
spark: SparkSession = org.apache.spark.sql.classic.SparkSession@3acccf6d
sc: org.apache.spark.SparkContext = org.apache.spark.SparkContext@417613ba

In [4]:
// Creamos un RDD a partir de una lista de Scala
// parallelize() distribuye la colección en particiones
val numeros = sc.parallelize(List(1, 2, 3, 4, 5, 6, 7, 8, 9, 10))

// Inspeccionamos el RDD (esto NO es una acción todavía)
println(s"Tipo del RDD:       ${numeros.getClass.getSimpleName}")
println(s"Número de partes:   ${numeros.getNumPartitions}")
println(s"¿Se ha ejecutado?:  No — solo hemos definido el plan")

Tipo del RDD:       ParallelCollectionRDD
Número de partes:   8
¿Se ha ejecutado?:  No — solo hemos definido el plan


numeros: org.apache.spark.rdd.RDD[Int] = ParallelCollectionRDD[0] at parallelize at cmd4.sc:3

In [5]:
// Transformación 1: filtrar solo los números pares
// Esta línea NO ejecuta nada — solo define el plan
val pares = numeros.filter(n => n % 2 == 0)

// Transformación 2: calcular el cuadrado de cada número par
// Tampoco ejecuta nada — extiende el plan anterior
val cuadradosDePares = pares.map(n => n * n)

println("Transformaciones definidas (plan listo, pero aún sin ejecutar):")
println(s"  numeros            → ${numeros.toDebugString.split('\n').head}")
println(s"  pares              → filtrar pares")
println(s"  cuadradosDePares   → elevar al cuadrado")
println()
println("Para ejecutar el plan, necesitamos una ACCIÓN...")

Transformaciones definidas (plan listo, pero aún sin ejecutar):
  numeros            → (8) ParallelCollectionRDD[0] at parallelize at cmd4.sc:3 []
  pares              → filtrar pares
  cuadradosDePares   → elevar al cuadrado

Para ejecutar el plan, necesitamos una ACCIÓN...


pares: org.apache.spark.rdd.RDD[Int] = MapPartitionsRDD[1] at filter at cmd5.sc:3
cuadradosDePares: org.apache.spark.rdd.RDD[Int] = MapPartitionsRDD[2] at map at cmd5.sc:7

In [6]:
// collect() es una ACCIÓN — aquí sí se ejecuta todo el plan
// ⚠️ En producción, collect() solo se usa con datasets pequeños
//    porque trae todos los datos al Driver (memoria del notebook)
val resultado = cuadradosDePares.collect()

println("✅ Acción collect() ejecutada — el job ha terminado")
println(s"Cuadrados de los números pares del 1 al 10:")
println(resultado.mkString(", "))

✅ Acción collect() ejecutada — el job ha terminado
Cuadrados de los números pares del 1 al 10:
4, 16, 36, 64, 100


resultado: Array[Int] = Array(4, 16, 36, 64, 100)

In [10]:
val datos = sc.parallelize(List(10, 20, 30, 40, 50, 60, 70, 80, 90, 100))

// count() devuelve el número de elementos (una acción)
val total = datos.count()

// sum() calcula la suma de todos los elementos
val suma = datos.sum()

// reduce() aplica una función binaria de forma distribuida
// Aquí calculamos el máximo: de dos elementos, quedamos con el mayor
val maximo = datos.reduce((a, b) => if (a > b) a else b)

// first() devuelve el primer elemento (sin traer todo al Driver)
val primero = datos.first()

println(s"Número de elementos: $total")
println(s"Suma total:          $suma")
println(s"Valor máximo:        $maximo")
println(s"Primer elemento:     $primero")

Número de elementos: 10
Suma total:          550.0
Valor máximo:        100
Primer elemento:     10


datos: org.apache.spark.rdd.RDD[Int] = ParallelCollectionRDD[35] at parallelize at cmd10.sc:1
total: Long = 10L
suma: Double = 550.0
maximo: Int = 100
primero: Int = 10

In [ ]:
//*************************************************

// Datos de ejemplo: un RDD de líneas de texto
val lineas = sc.parallelize(List(
  "apache spark es un motor de procesamiento distribuido",
  "spark procesa datos en memoria de forma muy eficiente",
  "scala es el lenguaje nativo de apache spark",
  "spark tiene apis en scala python java y r"
))

// Paso 1: dividir cada línea en palabras (flatMap aplana la lista de listas)
val palabras = lineas.flatMap(linea => linea.split(" "))

// Paso 2: convertir cada palabra en un par (palabra, 1)
// Los pares (K, V) son la base del modelo MapReduce
val pares = palabras.map(palabra => (palabra, 1))

// Paso 3: sumar los valores agrupados por clave (palabra)
// reduceByKey aplica la función a todos los valores con la misma clave
val conteo = pares.reduceByKey((a, b) => a + b)

// Paso 4: ordenar por conteo descendente (acción implícita en sortBy)
val ordenado = conteo.sortBy({ case (_, count) => count }, ascending = false)

// Acción: traer los resultados al Driver
val top10 = ordenado.take(10)  // take(N) es más eficiente que collect() para los primeros N

println("Top 10 palabras más frecuentes:")
println("─" * 35)
top10.foreach { case (palabra, count) =>
  println(f"  $palabra%-30s → $count veces")
}

In [14]:
import org.apache.spark.sql.SparkSession
import org.apache.log4j.{Level, Logger}

// 1. Detenemos cualquier sesión previa "contaminada"
SparkSession.active.stop()

// 2. Creamos la sesión bloqueando a Kryo
val spark = SparkSession.builder()
  .master("local[*]")
  .appName("WordCountCorrecto")
  // ESTA ES LA CLAVE: Forzamos el uso del serializador de Java, no Kryo
  .config("spark.serializer", "org.apache.spark.serializer.JavaSerializer")
  .config("spark.kryo.registrationRequired", "false")
  .getOrCreate()

val sc = spark.sparkContext
Logger.getLogger("org").setLevel(Level.ERROR)

println("✅ SparkSession lista (Serializador Java forzado)")

// --- AQUÍ VA TU CÓDIGO DE LAS LÍNEAS ---

✅ SparkSession lista (Serializador Java forzado)


import org.apache.spark.sql.SparkSession
import org.apache.log4j.{Level, Logger}
spark: SparkSession = org.apache.spark.sql.classic.SparkSession@46794ae4
sc: org.apache.spark.SparkContext = org.apache.spark.SparkContext@703a9907

In [15]:
// 3. Tu código (ligeramente optimizado para evitar shuffles innecesarios)
val lineas = sc.parallelize(List(
  "apache spark es un motor de procesamiento distribuido",
  "spark procesa datos en memoria de forma muy eficiente",
  "scala es el lenguaje nativo de apache spark",
  "spark tiene apis en scala python java y r"
))

val conteo = lineas
  .flatMap(_.split(" "))
  .map((_, 1))
  .reduceByKey(_ + _)
  .sortBy(_._2, ascending = false)

// Acción final
println("\n✅ ¡ÉXITO! Top 10 palabras:")
println("─" * 35)
conteo.take(10).foreach { case (palabra, count) =>
  println(f"  $palabra%-30s → $count veces")
}

java.lang.IllegalArgumentException: Unable to create serializer "com.esotericsoftware.kryo.serializers.FieldSerializer" for class: java.nio.HeapByteBuffer

In [16]:
import org.apache.spark.storage.StorageLevel

// RDD que simula un dataset de ventas con procesamiento costoso
val ventas = sc.parallelize(1 to 1_000_000)
  .map(id => (id, scala.util.Random.nextInt(1000), scala.util.Random.nextInt(50)))
  // (id, importe, region_id)

// Indicamos a Spark que guarde este RDD en memoria cuando se calcule por primera vez
// MEMORY_AND_DISK: si no cabe en RAM, desborda a disco (más seguro)
ventas.persist(StorageLevel.MEMORY_AND_DISK)

// Primera acción: Spark calcula el RDD y lo guarda en caché
val totalVentas = ventas.count()
println(s"Total de ventas: $totalVentas")

// Segunda acción: Spark REUTILIZA el RDD cacheado (mucho más rápido)
val importeTotal = ventas.map { case (_, importe, _) => importe }.sum()
println(f"Importe total: $importeTotal%.0f")

// Cuando ya no necesitamos el RDD, liberamos la memoria
ventas.unpersist()
println("✅ Caché liberado")

Total de ventas: 1000000
Importe total: 499377302
✅ Caché liberado


import org.apache.spark.storage.StorageLevel
ventas: org.apache.spark.rdd.RDD[(Int, Int, Int)] = MapPartitionsRDD[8] at map at cmd16.sc:5
res16_2: org.apache.spark.rdd.RDD[(Int, Int, Int)] = MapPartitionsRDD[8] at map at cmd16.sc:5
totalVentas: Long = 1000000L
importeTotal: Double = 4.99377302E8
res16_7: org.apache.spark.rdd.RDD[(Int, Int, Int)] = MapPartitionsRDD[8] at map at cmd16.sc:5

In [17]:
// Construimos un pipeline de varias transformaciones
val pipeline = sc.parallelize(1 to 20)
  .filter(_ % 2 == 0)           // solo pares
  .map(n => n * n)              // elevar al cuadrado
  .filter(_ > 50)               // solo los mayores que 50
  .map(n => s"resultado: $n")   // convertir a string

// toDebugString muestra el grafo de dependencias (linaje del RDD)
println("Linaje del RDD (de abajo a arriba = del origen al resultado):")
println("─" * 60)
println(pipeline.toDebugString)
println("─" * 60)
println()

// Ejecutamos la acción para ver el resultado
val resultados = pipeline.collect()
println(s"Resultados (${resultados.length} elementos):")
resultados.foreach(println)

Linaje del RDD (de abajo a arriba = del origen al resultado):
────────────────────────────────────────────────────────────
(8) MapPartitionsRDD[15] at map at cmd17.sc:6 []
 |  MapPartitionsRDD[14] at filter at cmd17.sc:5 []
 |  MapPartitionsRDD[13] at map at cmd17.sc:4 []
 |  MapPartitionsRDD[12] at filter at cmd17.sc:3 []
 |  ParallelCollectionRDD[11] at parallelize at cmd17.sc:2 []
────────────────────────────────────────────────────────────

Resultados (7 elementos):
resultado: 64
resultado: 100
resultado: 144
resultado: 196
resultado: 256
resultado: 324
resultado: 400


pipeline: org.apache.spark.rdd.RDD[String] = MapPartitionsRDD[15] at map at cmd17.sc:6
resultados: Array[String] = Array(
  "resultado: 64",
  "resultado: 100",
  "resultado: 144",
  "resultado: 196",
  "resultado: 256",
  "resultado: 324",
  "resultado: 400"
)

In [18]:
// Datos de ejemplo: un RDD de líneas de texto
val lineas = sc.parallelize(List(
  "apache spark es un motor de procesamiento distribuido",
  "spark procesa datos en memoria de forma muy eficiente",
  "scala es el lenguaje nativo de apache spark",
  "spark tiene apis en scala python java y r"
))

// Paso 1: dividir cada línea en palabras (flatMap aplana la lista de listas)
val palabras = lineas.flatMap(linea => linea.split(" "))

// Paso 2: convertir cada palabra en un par (palabra, 1)
// Los pares (K, V) son la base del modelo MapReduce
val pares = palabras.map(palabra => (palabra, 1))

// Paso 3: sumar los valores agrupados por clave (palabra)
// reduceByKey aplica la función a todos los valores con la misma clave
val conteo = pares.reduceByKey((a, b) => a + b)

// Paso 4: ordenar por conteo descendente (acción implícita en sortBy)
val ordenado = conteo.sortBy({ case (_, count) => count }, ascending = false)

// Acción: traer los resultados al Driver
val top10 = ordenado.take(10)  // take(N) es más eficiente que collect() para los primeros N

println("Top 10 palabras más frecuentes:")
println("─" * 35)
top10.foreach { case (palabra, count) =>
  println(f"  $palabra%-30s → $count veces")
}

java.lang.IllegalArgumentException: Unable to create serializer "com.esotericsoftware.kryo.serializers.FieldSerializer" for class: java.nio.HeapByteBuffer

In [20]:
import org.apache.spark.sql.SparkSession
import org.apache.spark.sql.functions._
import org.apache.log4j.{Level, Logger}

// 1. Configuramos los logs para que no estorben
Logger.getLogger("org").setLevel(Level.ERROR)

// 2. Creamos la sesión (usando la que ya definiste o creando una nueva)
val spark = SparkSession.builder()
  .appName("WordCountSolucion")
  .master("local[*]")
  .getOrCreate()

import spark.implicits._

// 3. Definimos los datos
val datos = List(
  "apache spark es un motor de procesamiento distribuido",
  "spark procesa datos en memoria de forma muy eficiente",
  "scala es el lenguaje nativo de apache spark",
  "spark tiene apis en scala python java y r"
)

// 4. Convertimos a DataFrame directamente (esto evita los problemas de RDD)
val df = datos.toDF("linea")

// 5. Procesamiento: 
//    - split: divide la línea en array de palabras
//    - explode: convierte el array en filas individuales
//    - groupBy y count: hace el trabajo de reduceByKey
//    - orderBy: hace el trabajo de sortBy
val resultado = df
  .select(explode(split($"linea", " ")).as("palabra"))
  .groupBy("palabra")
  .count()
  .orderBy($"count".desc)

// 6. MOSTRAR EL RESULTADO
println("\n✅ Top 10 palabras más frecuentes:")
println("─" * 35)
resultado.show(10)


✅ Top 10 palabras más frecuentes:
───────────────────────────────────
+-------------+-----+
|      palabra|count|
+-------------+-----+
|        spark|    4|
|           de|    3|
|           es|    2|
|       apache|    2|
|           en|    2|
|        scala|    2|
|procesamiento|    1|
|  distribuido|    1|
|        motor|    1|
|           un|    1|
+-------------+-----+
only showing top 10 rows


import org.apache.spark.sql.SparkSession
import org.apache.spark.sql.functions._
import org.apache.log4j.{Level, Logger}
spark: SparkSession = org.apache.spark.sql.classic.SparkSession@46794ae4
import spark.implicits._
datos: List[String] = List(
  "apache spark es un motor de procesamiento distribuido",
  "spark procesa datos en memoria de forma muy eficiente",
  "scala es el lenguaje nativo de apache spark",
  "spark tiene apis en scala python java y r"
)
df: org.apache.spark.sql.package.DataFrame = [linea: string]
resultado: org.apache.spark.sql.Dataset[org.apache.spark.sql.Row] = [palabra: string, count: bigint]

In [21]:
spark.stop()
println("✅ SparkSession cerrada correctamente")
println("   El Spark UI (localhost:4040) ya no está disponible")

✅ SparkSession cerrada correctamente
   El Spark UI (localhost:4040) ya no está disponible
